In [3]:
from pyspark.sql.functions import col, trim, upper, lit, current_timestamp, input_file_name
from pyspark.sql.types import StructType
from delta.tables import DeltaTable

# Path config - all paths relative to attached lakehouse
# BASE_PATH = "abfss://c5100bc6-a24c-467c-a9be-6b52c6eddade@onelake.dfs.fabric.microsoft.com/937f48bd-9cb3-4dfd-8c02-1535003395b6"

BRONZE_PATH = "Files/bronze/ae_monthly"
BRONZE_TABLE = "Table/bronze_ae_monthly"

print("Imports done")

StatementMeta(, 9a286d88-196f-4781-82c2-0f4402430d18, 5, Finished, Available, Finished, False)

Imports done


In [4]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false") # Keep everything as string at bronze, no casting (no autodetects column type)
    .option("recursiveFileLookup", "true") # Traverse all nested folders
    .csv(BRONZE_PATH)
    .withColumn("source_file", input_file_name()) # track which file each row comes from
    .withColumn("ingested_at", current_timestamp()) # audit timestamp
)

print(f"Total rows: {df_raw.count()}")
print(f"Total columns: {len(df_raw.columns)}")
print(f"Columns: {df_raw.columns}")
df_raw.printSchema()

StatementMeta(, 9a286d88-196f-4781-82c2-0f4402430d18, 6, Finished, Available, Finished, False)

Total rows: 7405
Total columns: 24
Columns: ['Period', 'Org Code', 'Parent Org', 'Org name', 'A&E attendances Type 1', 'A&E attendances Type 2', 'A&E attendances Other A&E Department', 'A&E attendances Booked Appointments Type 1', 'A&E attendances Booked Appointments Type 2', 'A&E attendances Booked Appointments Other Department', 'Attendances over 4hrs Type 1', 'Attendances over 4hrs Type 2', 'Attendances over 4hrs Other Department', 'Attendances over 4hrs Booked Appointments Type 1', 'Attendances over 4hrs Booked Appointments Type 2', 'Attendances over 4hrs Booked Appointments Other Department', 'Patients who have waited 4-12 hs from DTA to admission', 'Patients who have waited 12+ hrs from DTA to admission', 'Emergency admissions via A&E - Type 1', 'Emergency admissions via A&E - Type 2', 'Emergency admissions via A&E - Other A&E department', 'Other emergency admissions', 'source_file', 'ingested_at']
root
 |-- Period: string (nullable = true)
 |-- Org Code: string (nullable = true)

In [5]:
from pyspark.sql.functions import count, when, isnull, regexp_extract

print("=== NULL COUNTS PER COLUMN ===")
null_counts = df_raw.select([
    count(when(isnull(c), c)).alias(c)
    for c in df_raw.columns
    if c not in ("source_file", "ingested_at")
])
null_counts.show(vertical=True)

print("=== ROW COUNT PER YEAR ===")
df_raw.withColumn(
    "year",
    regexp_extract(col("source_file"), r"(20\d{2}-\d{2})", 1)
).groupBy("year").count().orderBy("year").show()

print("=== SAMPLE ROWS ===")
df_raw.select(
    "Period", "Org Code", "Org name", "A&E attendances Type 1"
).show(5, truncate=False)

StatementMeta(, 9a286d88-196f-4781-82c2-0f4402430d18, 7, Finished, Available, Finished, False)

=== NULL COUNTS PER COLUMN ===
-RECORD 0---------------------------------------------------------
 Period                                                     | 0   
 Org Code                                                   | 0   
 Parent Org                                                 | 0   
 Org name                                                   | 0   
 A&E attendances Type 1                                     | 0   
 A&E attendances Type 2                                     | 0   
 A&E attendances Other A&E Department                       | 0   
 A&E attendances Booked Appointments Type 1                 | 0   
 A&E attendances Booked Appointments Type 2                 | 0   
 A&E attendances Booked Appointments Other Department       | 0   
 Attendances over 4hrs Type 1                               | 0   
 Attendances over 4hrs Type 2                               | 0   
 Attendances over 4hrs Other Department                     | 0   
 Attendances over 4hrs Booked A

In [9]:
# Rename all columns to snake_case for Delta compatibility
column_mapping = {
    "Period": "period",
    "Org Code": "org_code",
    "Parent Org": "parent_org",
    "Org name": "org_name",
    "A&E attendances Type 1": "ae_attendances_type1",
    "A&E attendances Type 2": "ae_attendances_type2",
    "A&E attendances Other A&E Department": "ae_attendances_other",
    "A&E attendances Booked Appointments Type 1": "ae_attendances_booked_type1",
    "A&E attendances Booked Appointments Type 2": "ae_attendances_booked_type2",
    "A&E attendances Booked Appointments Other Department": "ae_attendances_booked_other",
    "Attendances over 4hrs Type 1": "attendance_over_4hrs_type1",
    "Attendances over 4hrs Type 2": "attendance_over_4hrs_type2",
    "Attendances over 4hrs Other Department": "attendance_over_4hrs_other",
    "Attendances over 4hrs Booked Appointments Type 1": "attendance_over_4hrs_booked_type1",
    "Attendances over 4hrs Booked Appointments Type 2": "attendance_over_4hrs_booked_type2",
    "Attendances over 4hrs Booked Appointments Other Department": "attendance_over_4hrs_booked_other",
    "Patients who have waited 4-12 hs from DTA to admission": "patient_waited_4_12hrs_dta",
    "Patients who have waited 12+ hrs from DTA to admission": "patient_waited_12plus_hrs_dta",
    "Emergency admissions via A&E - Type 1": "emergency_admissions_type1",
    "Emergency admissions via A&E - Type 2": "emergency_admissions_type2",
    "Emergency admissions via A&E - Other A&E department": "emergency_admissions_other",
    "Other emergency admissions": "other_emergency_admissions",
    "source_file": "source_file",
    "ingested_at": "ingested_at"
}

df_bronze = df_raw
for old_name, new_name in column_mapping.items():
    df_bronze = df_bronze.withColumnRenamed(old_name, new_name)

print("✅ Columns renamed")
print(df_bronze.columns)

StatementMeta(, 9a286d88-196f-4781-82c2-0f4402430d18, 11, Finished, Available, Finished, False)

✅ Columns renamed
['period', 'org_code', 'parent_org', 'org_name', 'ae_attendances_type1', 'ae_attendances_type2', 'ae_attendances_other', 'ae_attendances_booked_type1', 'ae_attendances_booked_type2', 'ae_attendances_booked_other', 'attendance_over_4hrs_type1', 'attendance_over_4hrs_type2', 'attendance_over_4hrs_other', 'attendance_over_4hrs_booked_type1', 'attendance_over_4hrs_booked_type2', 'attendance_over_4hrs_booked_other', 'patient_waited_4_12hrs_dta', 'patient_waited_12plus_hrs_dta', 'emergency_admissions_type1', 'emergency_admissions_type2', 'emergency_admissions_other', 'other_emergency_admissions', 'source_file', 'ingested_at']


In [10]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_ae_monthly")
)

print(f"✅ Bronze Delta table written")
print(f"Row Count: {spark.read.table('bronze_ae_monthly').count()}")

StatementMeta(, 9a286d88-196f-4781-82c2-0f4402430d18, 12, Finished, Available, Finished, False)

✅ Bronze Delta table written
Row Count: 7405
